# Détection visuelle des inondations — comparaison propre des 4 architectures

Version corrigée de `flood_comparaison.ipynb`, pour la version finale de l'article (AKTIION 2026).
**À exécuter sur Colab avec GPU** (T4 : ~1 h 30 pour 4 modèles × 5 plis ; reprise automatique si
la session se coupe).

| Problème dans `flood_comparaison.ipynb` | Correction ici |
|---|---|
| Doublons (dont exacts) entre `/train` et `/val` ; 472 originales et non 410 | Départ des **images originales classées à la main** (`datasetflood/flooded`, `datasetflood/not_flooded`) ; empreintes pHash/dHash : doublons exacts retirés, quasi-doublons **toujours dans le même pli** |
| Les 113 images de `/val` servent à l'arrêt anticipé, au choix du checkpoint **et** à l'évaluation finale (le score publié = meilleur score de validation) | **Validation croisée à 5 plis** : chaque pli de test n'est jamais vu ; arrêt anticipé sur une **validation interne** tirée de l'entraînement |
| Une seule partition de 113 images | Chaque image est testée une fois → estimation sur tout le corpus, moyenne ± écart-type |
| Phase 1 « backbone gelé » non gelée pour ConvNeXt et Swin (mot-clé `'fc'` qui attrape `mlp.fc1/fc2`) | Gel via `get_classifier()` + **dernier étage défini explicitement** par architecture ; nombres de paramètres affichés |
| Grad-CAM de Swin faux (tenseurs `(B,H,W,C)` lus comme `(B,C,H,W)`) | Format géré explicitement, forme vérifiée |
| F1 « pondéré », classe positive ambiguë | Métriques de la **classe inondation** + F1 macro, AUC sur la probabilité d'inondation |
| Composition du jeu déduite des métriques | Comptage direct + provenance des images |

L'augmentation hors-ligne (copies sur disque) est remplacée par la même augmentation **à la
volée** + Mixup/CutMix : même effet, et aucune copie ne peut fuiter vers le test.

In [ ]:
!pip -q install timm imagehash
import os, re, json, time, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm, imagehash
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, precision_score,
                             recall_score, f1_score, accuracy_score)
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device, "| torch", torch.__version__, "| timm", timm.__version__)
assert device.type == "cuda", "Active le GPU : Exécution > Modifier le type d'exécution > GPU"


## 1. Configuration

In [ ]:
ROOT = "/content/drive/MyDrive/memoire/donnee/inondation_senegal"
DATA_DIR = f"{ROOT}/datasetflood"            # images originales classées à la main
OLD_DIR  = f"{ROOT}/flood_dataset"           # ancienne partition train/val (pour comparaison)
CLASS_DIRS = {"flooded": 1, "not_flooded": 0}  # 1 = inondation (classe positive)
OUT_DIR  = "/content/drive/MyDrive/memoire/cnn_propre"     # résultats (reprise possible)
CKPT_DIR = "/content/ckpt"                                  # checkpoints temporaires (hors Drive)
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)

N_FOLDS = 5
SEEDS = [42]                 # [42, 1, 2] si le temps le permet (≈ 1 h 30 par graine)
NEAR_DUP_DIST = 8            # distance de Hamming pHash pour « quasi-doublon »
IMG = 224
CLS_NAMES = {0: "normale", 1: "inondation"}

CFG = dict(batch_size=16, p1_epochs=5, p2_epochs=8, p3_epochs=12, p1_lr=1e-3, p2_lr=1e-4, p3_lr=3e-5,
           weight_decay=0.02, patience=7, mixup_alpha=0.4, cutmix_alpha=1.0, inner_val_frac=0.15)

MODELS = {   # nom -> (nom timm, fonction qui renvoie le dernier étage dégelé en phase 2)
    "ResNet-18":       ("resnet18",                     lambda m: [m.layer4]),
    "EfficientNet-B3": ("efficientnet_b3",              lambda m: [m.blocks[-2:], m.conv_head, m.bn2]),
    "ConvNeXt-Tiny":   ("convnext_tiny",                lambda m: [m.stages[-1], m.head]),
    "Swin-Tiny":       ("swin_tiny_patch4_window7_224", lambda m: [m.layers[-1], m.norm]),
}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

## 2. Les images originales

Lecture de `datasetflood/flooded` et `datasetflood/not_flooded`, provenance d'après le nom de
fichier, puis comparaison avec l'ancienne partition `flood_dataset/` (train sans les copies
`aug_`, et val) pour mesurer ce qui s'y était passé.

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def provenance(n):
    s = re.sub(r"\.(jpe?g|png|webp|bmp)$", "", n, flags=re.I)
    return ("téléchargée (nom par défaut du navigateur)" if re.match(r"^(images|téléchargement)", s)
            else "renommée par l'auteur (citoyen / terrain)")

recs = []
for d, lab in CLASS_DIRS.items():
    for p in sorted(Path(DATA_DIR, d).iterdir()):
        if p.suffix.lower() in EXT:
            recs.append(dict(path=str(p), name=p.name, folder=d, label=lab, label_name=CLS_NAMES[lab]))
orig = pd.DataFrame(recs)
orig["provenance"] = orig.name.map(provenance)
print(f"{len(orig)} images originales")
print(pd.crosstab(orig.provenance, orig.label_name, margins=True))

def hashes(p):
    im = Image.open(p).convert("RGB")
    return imagehash.phash(im), imagehash.dhash(im)
H = [hashes(p) for p in orig.path]
orig.to_csv(f"{OUT_DIR}/images_originales.csv", index=False)

In [ ]:
# Comparaison avec l'ancienne partition flood_dataset/ (train sans les aug_, et val)
old = []
for split in ["train", "val"]:
    for cls, lab in [("inondation", 1), ("non_inondation", 0)]:
        for p in Path(OLD_DIR, split, cls).iterdir():
            if p.suffix.lower() in EXT and not p.name.startswith("aug_"):
                old.append(dict(path=str(p), split=split, label=lab))
old = pd.DataFrame(old)
ph_orig = [h[0] for h in H]
def closest(p):
    h = imagehash.phash(Image.open(p).convert("RGB"))
    d = [h - x for x in ph_orig]; i = int(np.argmin(d))
    return d[i], i
res = [closest(p) for p in old.path]
old["dist"] = [r[0] for r in res]; old["orig_idx"] = [r[1] for r in res]
old["meme_label"] = old.label.values == orig.label.values[old.orig_idx]
found = old[old.dist <= 4]
print("Ancienne partition (hors aug_) :", old.groupby(["split", "label"]).size().to_dict())
print("  retrouvées dans les originaux (distance <= 4) :", len(found), "/", len(old))
print("  retrouvées avec une étiquette différente     :", int((~found.meme_label).sum()))
print("  originaux présents plusieurs fois            :", int(found.orig_idx.duplicated().sum()))
leak = set(found[found.split == "val"].orig_idx) & set(found[found.split == "train"].orig_idx)
print("  images de val/ dont l'original est aussi dans train/ :", len(leak))
old.to_csv(f"{OUT_DIR}/ancienne_partition_vs_originaux.csv", index=False)

## 3. Doublons et quasi-doublons

Images à distance ≤ `NEAR_DUP_DIST` (pHash, et dHash ≤ `NEAR_DUP_DIST` + 4) : reliées en
**groupes** qui ne seront jamais séparés entre entraînement et test. Les **doublons exacts**
(distance 0, même étiquette) sont retirés ; les doublons à **étiquettes contradictoires** sont
signalés pour vérification manuelle.

In [ ]:
parent = list(range(len(orig)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
pairs = []
for i in range(len(orig)):
    for j in range(i + 1, len(orig)):
        dp, dd = H[i][0] - H[j][0], H[i][1] - H[j][1]
        if dp <= NEAR_DUP_DIST and dd <= NEAR_DUP_DIST + 4:
            pairs.append((i, j, dp))
            parent[find(i)] = find(j)
orig["group"] = [find(i) for i in range(len(orig))]

pp = pd.DataFrame([dict(a=orig.name[i], classe_a=orig.folder[i], b=orig.name[j], classe_b=orig.folder[j],
                        dist=d, meme_label=orig.label[i] == orig.label[j]) for i, j, d in pairs])
print(f"{len(pp)} paires de (quasi-)doublons, dont {int((pp.dist == 0).sum()) if len(pp) else 0} exactes")
if len(pp):
    print(f"  étiquettes contradictoires : {int((~pp.meme_label).sum())}")
    pp.to_csv(f"{OUT_DIR}/doublons.csv", index=False)
    display(pp.sort_values("dist").head(30))

In [ ]:
# Aperçu visuel des 8 paires les plus proches (à vérifier à l'œil)
if pairs:
    k8 = sorted(pairs, key=lambda x: x[2])[:8]
    fig, axes = plt.subplots(2, len(k8), figsize=(2.2 * len(k8), 4.6), squeeze=False)
    for k, (i, j, d) in enumerate(k8):
        for r, idx in enumerate((i, j)):
            axes[r, k].imshow(Image.open(orig.path[idx]).convert("RGB")); axes[r, k].axis("off")
            axes[r, k].set_title(f"{orig.folder[idx]} d={d}", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
# Retrait des doublons exacts de même étiquette (on garde une copie par groupe exact)
drop = {j for i, j, d in pairs if d == 0 and orig.label[i] == orig.label[j]}
contra = [(orig.name[i], orig.name[j]) for i, j, d in pairs if orig.label[i] != orig.label[j]]
if contra:
    print("⚠️ Images proches avec des étiquettes différentes (à vérifier, retirées par prudence) :", contra)
    drop |= {j for i, j, d in pairs if orig.label[i] != orig.label[j]} | {i for i, j, d in pairs if orig.label[i] != orig.label[j]}
data = orig.drop(index=sorted(drop)).reset_index(drop=True)
print(f"images retenues : {len(data)} (retirées : {len(drop)})")
print(data.label_name.value_counts())
print("groupes de plus d'une image :", int((data.group.value_counts() > 1).sum()))
data.to_csv(f"{OUT_DIR}/images_retenues.csv", index=False)

### 3 bis. Tableau et exemples du corpus (demande du relecteur AKTIION)

Le relecteur demande « un tableau des données utilisées et une extraction qui les illustre ».
On produit : le tableau classe × provenance avec la taille des images, et une planche
d'exemples (anglais et français) pour l'article.

In [ ]:
sizes = [Image.open(p).size for p in data.path]
data["largeur"], data["hauteur"] = [s[0] for s in sizes], [s[1] for s in sizes]
tab_data = data.groupby(["label_name", "provenance"]).agg(images=("path", "size"),
            largeur_med=("largeur", "median"), hauteur_med=("hauteur", "median")).reset_index()
tab_data.to_csv(f"{OUT_DIR}/tableau_donnees.csv", index=False)
display(tab_data)
print("Total :", len(data), "images |", data.label_name.value_counts().to_dict())

rng_ex = np.random.default_rng(7)
EX = {"en": ("Flood", "No flood"), "fr": ("Inondation", "Normale")}
for lang, (t1, t0) in EX.items():
    fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
    for r, lab in enumerate([1, 0]):
        idx = rng_ex.choice(np.where(data.label == lab)[0], 6, replace=False)
        for c, i in enumerate(idx):
            axes[r, c].imshow(Image.open(data.path[i]).convert("RGB").resize((IMG, IMG))); axes[r, c].axis("off")
        axes[r, 0].set_title(t1 if lab == 1 else t0, loc="left", fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/exemples_corpus_{lang}.png", dpi=150); plt.show()

## 4. Partition en 5 plis (stratifiée, par groupes)

Chaque image est dans le test d'exactement un pli. Dans chaque pli, ~15 % de l'entraînement (par
groupes) sert de **validation interne** pour l'arrêt anticipé et le choix du checkpoint.

In [ ]:
def make_splits(seed):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    folds = []
    for k, (trv, te) in enumerate(sgkf.split(data, data.label, data.group)):
        inner = StratifiedGroupKFold(n_splits=round(1 / CFG["inner_val_frac"]), shuffle=True, random_state=seed + k)
        tr_i, va_i = next(inner.split(data.iloc[trv], data.label.iloc[trv], data.group.iloc[trv]))
        folds.append(dict(train=trv[tr_i], val=trv[va_i], test=te))
    return folds

SPLITS = {s: make_splits(s) for s in SEEDS}
rows = []
for s, folds in SPLITS.items():
    for k, f in enumerate(folds):
        assert not set(data.group.iloc[f["test"]]) & set(data.group.iloc[np.r_[f["train"], f["val"]]])
        assert not set(data.group.iloc[f["val"]]) & set(data.group.iloc[f["train"]])
        for part in ("train", "val", "test"):
            y = data.label.iloc[f[part]]
            rows.append(dict(seed=s, pli=k, partie=part, n=len(y), inondation=int(y.sum()), normale=int((1 - y).sum())))
split_tab = pd.DataFrame(rows)
split_tab.to_csv(f"{OUT_DIR}/partitions.csv", index=False)
split_tab.pivot_table(index=["seed", "pli"], columns="partie", values="n")

## 5. Données, modèles, gel des couches

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.Resize((256, 256)), T.RandomCrop((IMG, IMG)), T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.3),
    T.RandomRotation(30), T.RandomAffine(0, translate=(0.1, 0.1), scale=(0.85, 1.15), shear=10),
    T.ColorJitter(0.4, 0.4, 0.3, 0.1), T.RandomGrayscale(0.05), T.GaussianBlur(3, (0.1, 1.5)),
    T.RandomPerspective(0.2, p=0.3), T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.2, scale=(0.02, 0.1))])
eval_tf = T.Compose([T.Resize((IMG, IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])

IMCACHE = {p: Image.open(p).convert("RGB") for p in data.path}     # images en mémoire (Drive lent)

class Imgs(Dataset):
    def __init__(self, idx, tf):
        self.p = data.path.iloc[idx].tolist(); self.y = data.label.iloc[idx].tolist(); self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(IMCACHE[self.p[i]]), self.y[i]

def loader(idx, train):
    return DataLoader(Imgs(idx, train_tf if train else eval_tf), batch_size=CFG["batch_size"],
                      shuffle=train, drop_last=train, num_workers=2, pin_memory=True)

def build(name):
    tname, last_stage = MODELS[name]
    return timm.create_model(tname, pretrained=True, num_classes=2).to(device), last_stage

def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

def set_phase(m, last_stage, phase):
    for p in m.parameters():
        p.requires_grad = phase == 3
    if phase in (1, 2):
        for p in m.get_classifier().parameters():
            p.requires_grad = True
    if phase == 2:
        for mod in last_stage(m):
            for p in mod.parameters():
                p.requires_grad = True

# Vérification du gel : phase 1 = tête seule, phase 2 = + dernier étage, phase 3 = tout
for name in MODELS:
    m, ls = build(name)
    counts = []
    for ph in (1, 2, 3):
        set_phase(m, ls, ph); counts.append(n_trainable(m))
    print(f"{name:16s} phase1 {counts[0]:>11,} | phase2 {counts[1]:>11,} | phase3 {counts[2]:>11,}")
    assert counts[0] < 0.01 * counts[2] and counts[0] < counts[1] < counts[2]
    del m; torch.cuda.empty_cache()

## 6. Entraînement (3 phases, Mixup/CutMix, arrêt anticipé sur la validation interne)

In [ ]:
def mix(x, y, alpha, cut):
    lam = np.random.beta(alpha, alpha); idx = torch.randperm(x.size(0), device=x.device)
    if not cut:
        return lam * x + (1 - lam) * x[idx], y, y[idx], lam
    _, _, h, w = x.shape; r = np.sqrt(1 - lam); ch, cw = int(h * r), int(w * r)
    cy, cx = np.random.randint(h), np.random.randint(w)
    y1, y2 = np.clip(cy - ch // 2, 0, h), np.clip(cy + ch // 2, 0, h)
    x1, x2 = np.clip(cx - cw // 2, 0, w), np.clip(cx + cw // 2, 0, w)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    return x, y, y[idx], 1 - (y2 - y1) * (x2 - x1) / (h * w)

@torch.no_grad()
def predict(m, dl):
    m.eval(); P, Y = [], []
    for x, y in dl:
        P.append(torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def run_phase(m, dl_tr, dl_va, crit, lr, epochs, wd, best, ckpt, alpha):
    opt = optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=wd)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 100)
    wait = 0
    for ep in range(epochs):
        m.train()
        for x, y in dl_tr:
            x, y = x.to(device), y.to(device)
            if np.random.rand() < 0.5:                       # Mixup ou CutMix sur la moitié des lots
                cut = np.random.rand() < 0.5
                x, ya, yb, lam = mix(x, y, CFG["cutmix_alpha"] if cut else alpha, cut)
                out = m(x); loss = lam * crit(out, ya) + (1 - lam) * crit(out, yb)
            else:
                loss = crit(m(x), y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step()
        p, yv = predict(m, dl_va)
        score = roc_auc_score(yv, p) + 1e-3 * accuracy_score(yv, p >= 0.5)   # critère : AUC de la validation interne
        if score > best + 1e-4:
            best, wait = score, 0; torch.save(m.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= CFG["patience"]:
                break
    return best

def train_eval(name, fold, seed):
    set_seed(seed)
    m, ls = build(name)
    dl_tr, dl_va, dl_te = loader(fold["train"], True), loader(fold["val"], False), loader(fold["test"], False)
    y_tr = data.label.iloc[fold["train"]].values
    w = torch.tensor([len(y_tr) / (2 * (y_tr == c).sum()) for c in (0, 1)], dtype=torch.float, device=device)
    crit = nn.CrossEntropyLoss(weight=w)
    ckpt = f"{CKPT_DIR}/{name}_best.pt"; best = -1.0; t0 = time.time()
    phases = [(CFG["p1_lr"], CFG["p1_epochs"]), (CFG["p2_lr"], CFG["p2_epochs"]), (CFG["p3_lr"], CFG["p3_epochs"])]
    for ph, (lr, ep) in enumerate(phases, start=1):
        set_phase(m, ls, ph)
        best = run_phase(m, dl_tr, dl_va, crit, lr, ep, CFG["weight_decay"] / (2 if ph == 3 else 1),
                         best, ckpt, CFG["mixup_alpha"] if ph < 3 else 0.3)
        m.load_state_dict(torch.load(ckpt))          # chaque phase repart du meilleur état
    p, y = predict(m, dl_te)                          # test : jamais vu pendant l'entraînement
    return m, p, y, time.time() - t0

In [ ]:
# Boucle principale — reprise automatique (un fichier JSON par modèle × pli × graine sur le Drive)
for seed in SEEDS:
    for k, fold in enumerate(SPLITS[seed]):
        for name in MODELS:
            f = Path(OUT_DIR, f"pred_{name}_s{seed}_f{k}.json")
            if f.exists():
                continue
            m, p, y, dt = train_eval(name, fold, seed)
            json.dump(dict(model=name, seed=seed, fold=k, idx=fold["test"].tolist(), prob=p.tolist(),
                           y=y.tolist(), minutes=dt / 60), open(f, "w"))
            if k == 0 and seed == SEEDS[0]:
                torch.save(m.state_dict(), f"{OUT_DIR}/{name}_s{seed}_f0.pt")   # pour Grad-CAM (sur le Drive)
            print(f"seed {seed} pli {k} {name:16s} AUC test = {roc_auc_score(y, p):.3f} ({dt/60:.1f} min)")
            del m; torch.cuda.empty_cache()

## 7. Résultats

In [ ]:
R = [json.load(open(f)) for f in sorted(Path(OUT_DIR).glob("pred_*.json"))]
order = list(MODELS)
def metr(y, p, thr=0.5):
    y, p = np.asarray(y), np.asarray(p); h = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, h, labels=[0, 1]).ravel()
    return dict(Accuracy=accuracy_score(y, h), F1_inondation=f1_score(y, h), F1_macro=f1_score(y, h, average="macro"),
                Rappel_inondation=recall_score(y, h), Precision_inondation=precision_score(y, h, zero_division=0),
                AUC=roc_auc_score(y, p), FP=int(fp), FN=int(fn),
                Taux_FP=fp / max(fp + tn, 1))       # utilisé par l'analyse de fusion (§V-D de l'article)

per_fold = pd.DataFrame([dict(Modèle=r["model"], seed=r["seed"], pli=r["fold"], **metr(r["y"], r["prob"])) for r in R])
per_fold.to_csv(f"{OUT_DIR}/metriques_par_pli.csv", index=False)
cols = ["Accuracy", "F1_inondation", "F1_macro", "Rappel_inondation", "Precision_inondation", "AUC"]
agg = per_fold.groupby("Modèle")[cols].agg(["mean", "std"]).loc[order]
fmt = lambda m, c: (f"{agg.loc[m, (c, 'mean')]:.3f} ± {agg.loc[m, (c, 'std')]:.3f}" if c == "AUC"
                    else f"{100*agg.loc[m, (c, 'mean')]:.1f} ± {100*agg.loc[m, (c, 'std')]:.1f}")
tab = pd.DataFrame({c: [fmt(m, c) for m in order] for c in cols}, index=order)
tab.to_csv(f"{OUT_DIR}/tableau_article.csv")
print(f"Moyenne ± écart-type sur {N_FOLDS} plis × {len(SEEDS)} graine(s), {len(data)} images")
tab

In [ ]:
# Prédictions hors-pli réunies (chaque image testée une fois par graine)
pooled = {}
for name in order:
    rr = sorted([r for r in R if r["model"] == name], key=lambda r: (r["seed"], r["fold"]))
    pooled[name] = (np.concatenate([r["y"] for r in rr]), np.concatenate([r["prob"] for r in rr]))
pool_tab = pd.DataFrame({n: metr(*pooled[n]) for n in order}).T
pool_tab.to_csv(f"{OUT_DIR}/metriques_poolees.csv")
display(pool_tab.round(3))

# ConvNeXt vs Swin : différence d'AUC par pli + bootstrap sur les images
d = (per_fold[per_fold.Modèle == "ConvNeXt-Tiny"].set_index(["seed", "pli"]).AUC
     - per_fold[per_fold.Modèle == "Swin-Tiny"].set_index(["seed", "pli"]).AUC)
print("ΔAUC ConvNeXt − Swin par pli :", d.round(3).tolist())
rng = np.random.default_rng(0); yC, pC = pooled["ConvNeXt-Tiny"]; yS, pS = pooled["Swin-Tiny"]
assert (yC == yS).all()
bs = []
for _ in range(2000):
    i = rng.integers(0, len(yC), len(yC))
    if len(set(yC[i])) == 2:
        bs.append(roc_auc_score(yC[i], pC[i]) - roc_auc_score(yS[i], pS[i]))
print(f"ΔAUC ConvNeXt − Swin (bootstrap) = {np.mean(bs):+.3f} [{np.percentile(bs, 2.5):+.3f} ; {np.percentile(bs, 97.5):+.3f}]")

# Taille et vitesse (colonnes « params » et « temps » du tableau de l'article)
cost = []
for n in order:
    m, _ = build(n); m.eval(); x = torch.randn(1, 3, IMG, IMG, device=device)
    with torch.no_grad():
        for _ in range(10): m(x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): m(x)
        torch.cuda.synchronize()
    cost.append(dict(Modèle=n, Params_M=sum(p.numel() for p in m.parameters()) / 1e6,
                     Inference_ms_GPU=(time.time() - t0) / 50 * 1000,
                     Entrainement_min=np.mean([r["minutes"] for r in R if r["model"] == n])))
    del m; torch.cuda.empty_cache()
cost = pd.DataFrame(cost).set_index("Modèle").round(2)
cost.to_csv(f"{OUT_DIR}/cout_modeles.csv")
cost

In [ ]:
LAB = {"en": dict(fpr="False positive rate", tpr="True positive rate",
                  pred="Predicted", true="True", cls=["No flood", "Flood"]),
       "fr": dict(fpr="Taux de faux positifs", tpr="Taux de vrais positifs",
                  pred="Prédit", true="Réel", cls=["Normale", "Inondation"])}
for lang, L in LAB.items():
    plt.figure(figsize=(5.5, 5))
    for n in order:
        y, p = pooled[n]; f, t, _ = roc_curve(y, p)
        plt.plot(f, t, lw=2, label=f"{n} (AUC = {roc_auc_score(y, p):.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=1); plt.xlabel(L["fpr"]); plt.ylabel(L["tpr"]); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cnn_roc_cv_{lang}.png", dpi=200); plt.show()

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
    for ax, n in zip(axes, order):
        y, p = pooled[n]; cm = confusion_matrix(y, (p >= 0.5).astype(int), labels=[0, 1])
        ax.imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, v, ha="center", va="center", fontsize=13, color="white" if v > cm.max() / 2 else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(L["cls"]); ax.set_yticks([0, 1]); ax.set_yticklabels(L["cls"])
        ax.set_xlabel(L["pred"]); ax.set_ylabel(L["true"]); ax.set_title(n)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cnn_confusion_cv_{lang}.png", dpi=200); plt.show()

## 8. Grad-CAM corrigé

Modèles du pli 0, sur des images de **test** de ce pli. Pour Swin, la sortie du dernier étage
est au format `(B, H, W, C)` (timm ≥ 0.9) : remise en `(B, C, H, W)` avant le calcul.

In [ ]:
TARGET = {"ResNet-18": (lambda m: m.layer4, False),
          "EfficientNet-B3": (lambda m: m.conv_head, False),
          "ConvNeXt-Tiny": (lambda m: m.stages[-1], False),
          "Swin-Tiny": (lambda m: m.layers[-1], True)}          # True = sortie (B, H, W, C)

def gradcam(m, layer, x, channels_last):
    store = {}
    def fwd(mod, i, o):
        store["a"] = o
        o.register_hook(lambda g: store.__setitem__("g", g))   # gradient de la sortie de l'étage
    h = layer.register_forward_hook(fwd)
    m.eval(); out = m(x); c = out.argmax(1).item()
    m.zero_grad(); out[0, c].backward(); h.remove()
    a, g = store["a"].detach(), store["g"].detach()
    if channels_last:
        a, g = a.permute(0, 3, 1, 2), g.permute(0, 3, 1, 2)
    cam = torch.relu((g.mean((2, 3), keepdim=True) * a).sum(1)).squeeze()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.cpu().numpy(), c, torch.softmax(out, 1)[0, c].item()

s0 = SEEDS[0]; te = SPLITS[s0][0]["test"]
nets = {}
for n in order:
    m, _ = build(n); m.load_state_dict(torch.load(f"{OUT_DIR}/{n}_s{s0}_f0.pt")); nets[n] = m
shape = {}
with torch.no_grad():
    h = TARGET["Swin-Tiny"][0](nets["Swin-Tiny"]).register_forward_hook(lambda mod, i, o: shape.__setitem__("s", tuple(o.shape)))
    nets["Swin-Tiny"](torch.zeros(1, 3, IMG, IMG, device=device)); h.remove()
print("sortie du dernier étage de Swin :", shape["s"], "-> attendu (1, 7, 7, 768)")
assert shape["s"][-1] == 768

rng = np.random.default_rng(1)
pick = list(rng.choice([i for i in te if data.label[i] == 1], 3, replace=False)) + \
       list(rng.choice([i for i in te if data.label[i] == 0], 2, replace=False))
for lang, L in LAB.items():
    fig, axes = plt.subplots(len(pick), 5, figsize=(15, 3 * len(pick)))
    for r, i in enumerate(pick):
        im = IMCACHE[data.path[i]].resize((IMG, IMG)); arr = np.asarray(im)
        axes[r, 0].imshow(arr); axes[r, 0].set_title(f"{L['true']} : {L['cls'][data.label[i]]}", fontsize=9)
        x = eval_tf(IMCACHE[data.path[i]]).unsqueeze(0).to(device)
        for c, n in enumerate(order, start=1):
            layer, cl = TARGET[n]
            cam, k, conf = gradcam(nets[n], layer(nets[n]), x, cl)
            heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, (IMG, IMG))), cv2.COLORMAP_JET)[:, :, ::-1]
            axes[r, c].imshow((0.5 * heat + 0.5 * arr).astype(np.uint8))
            axes[r, c].set_title(f"{n}\n{L['cls'][k]} ({conf:.0%})", fontsize=8,
                                 color="green" if k == data.label[i] else "red")
        for ax in axes[r]: ax.axis("off")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cnn_gradcam_cv_{lang}.png", dpi=150); plt.show()

## 9. À m'envoyer

Dans `MyDrive/memoire/cnn_propre/` : `tableau_article.csv`, `metriques_par_pli.csv`,
`metriques_poolees.csv`, `cout_modeles.csv`, `tableau_donnees.csv`, `doublons.csv`,
`partitions.csv`, `ancienne_partition_vs_originaux.csv`, et les figures `exemples_corpus_*.png`,
`cnn_roc_cv_*.png`, `cnn_confusion_cv_*.png`, `cnn_gradcam_cv_*.png`
— ou simplement le notebook exécuté (Fichier > Télécharger > .ipynb).